# Sales Driver Analysis
### Original example by Danny Barren: *ML for Business Insights, Examples*

**The business question:** *What actually moves our weekly sales, and which levers (advertising, discounts, staffing, price) are worth pulling harder?*

Here the goal is less about predicting next week's number and more about **explaining** it. So we lead with a model whose answers you can read straight off the page:

- **Ridge regression** (linear regression with a built-in "don't overreact" brake): every driver gets a clear *"+1 unit → +$X sales"* number
- **HistGradientBoosting** as a cross-check: if it scores much better, some effects are curved (e.g. ad spend with diminishing returns) and we look at those shapes too

Then we turn the numbers into a **lever table** with return on spend, plus a simple **what-if scenario**.

**How to use it:** Edit only the **CONFIG** cell, then run top to bottom.
If `CSV_PATH = None`, a **synthetic** store-week dataset is generated for the demo.

In [ ]:
# ============================================================
# IMPORTS (scikit-learn 1.2+)
# ============================================================
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import KFold, RepeatedKFold, TimeSeriesSplit, cross_validate, GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import mean_absolute_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Libraries loaded.")

In [ ]:
# ============================================================
# CONFIG: THE ONLY CELL YOU NEED TO EDIT
# ============================================================
CSV_PATH = None                     # e.g. "/content/store_weeks.csv"; None = synthetic demo

ID_COLUMN = "store_id"
TIME_COLUMN = "week"                # set to None if your rows are not in time order
TARGET_COLUMN = "weekly_sales"      # in dollars

NUMERIC_COLUMNS = ["digital_ad_spend", "local_ad_spend", "discount_pct", "avg_price",
                   "competitor_price_gap_pct", "staff_hours", "store_size_sqft", "temperature_c"]
CATEGORICAL_COLUMNS = ["region", "promo_display", "holiday_week"]
DROP_COLUMNS = []

# Which drivers can management actually change? (the rest are context we control for)
CONTROLLABLE_LEVERS = ["digital_ad_spend", "local_ad_spend", "discount_pct", "avg_price", "staff_hours", "promo_display"]
SPEND_LEVERS = ["digital_ad_spend", "local_ad_spend"]   # levers measured in dollars spent (for return on spend)
GROSS_MARGIN = 0.40                 # share of each sales dollar kept as gross profit

# What-if scenario: % change to apply to each lever (e.g. +10% digital spend)
SCENARIO = {"digital_ad_spend": 0.10, "local_ad_spend": 0.0}

USE_TIME_SPLIT = True               # True = test on the most recent weeks (honest for time data)
DATA_SIZE_MODE = "auto"
TEST_SIZE = 0.20
RANDOM_STATE = 42

In [ ]:
# ============================================================
# LOAD DATA (or build synthetic store-weeks)
# ============================================================
def make_demo_sales(n_stores=40, n_weeks=15, seed=RANDOM_STATE):
    """Synthetic store-week sales. Story: digital ads work but with diminishing returns,
    local ads help a little, discounts lift volume, higher price hurts, staffing helps
    up to a point, bigger stores and holiday weeks sell more."""
    rng = np.random.default_rng(seed)
    stores = pd.DataFrame({"store_id": np.arange(1, n_stores + 1),
                           "region": rng.choice(["north", "south", "east", "west"], n_stores),
                           "store_size_sqft": np.round(rng.uniform(4_000, 20_000, n_stores), -2),
                           "store_effect": rng.normal(0, 2_500, n_stores)})
    df = stores.loc[stores.index.repeat(n_weeks)].reset_index(drop=True)
    df["week"] = np.tile(np.arange(1, n_weeks + 1), n_stores)
    n = len(df)
    df["digital_ad_spend"] = np.round(rng.gamma(3, 600, n), -1)
    df["local_ad_spend"] = np.round(rng.gamma(2, 400, n), -1)
    df["discount_pct"] = rng.choice([0, 0, 5, 10, 15, 20], n).astype(float)
    df["avg_price"] = np.round(rng.normal(24, 2.5, n), 2)
    df["competitor_price_gap_pct"] = np.round(rng.normal(0, 6, n), 1)
    df["staff_hours"] = np.round(df["store_size_sqft"] / 60 + rng.normal(0, 25, n))
    df["temperature_c"] = np.round(rng.normal(14, 9, n), 1)
    df["promo_display"] = rng.choice(["yes", "no"], n, p=[0.3, 0.7])
    df["holiday_week"] = np.where(np.isin(df["week"], [6, 13]), "yes", "no")
    df["weekly_sales"] = np.round(
        18_000 + 1.6 * df["store_size_sqft"] + 11_000 * np.log1p(df["digital_ad_spend"] / 1_000)
        + 1.2 * df["local_ad_spend"] + 260 * df["discount_pct"] - 900 * (df["avg_price"] - 24)
        + 120 * df["competitor_price_gap_pct"] + 40 * np.minimum(df["staff_hours"], 300)
        + 2_500 * (df["promo_display"] == "yes") + 6_000 * (df["holiday_week"] == "yes")
        + 1_500 * (df["region"] == "south") + df["store_effect"] + rng.normal(0, 2_000, n), 0)
    df = df.drop(columns="store_effect")
    df.loc[rng.random(n) < 0.03, "temperature_c"] = np.nan
    return df

if CSV_PATH is None:
    df = make_demo_sales()
    print("Using SYNTHETIC demo data (CSV_PATH is None).")
else:
    df = pd.read_csv(CSV_PATH)
    print(f"Loaded {CSV_PATH}")
print(f"{len(df):,} rows x {df.shape[1]} columns")
df.head()

In [ ]:
# ============================================================
# CONFIG CHECK
# ============================================================
needed = [TARGET_COLUMN] + NUMERIC_COLUMNS + CATEGORICAL_COLUMNS + ([TIME_COLUMN] if TIME_COLUMN else [])
missing = [c for c in needed if c not in df.columns]
assert not missing, f"These CONFIG columns are not in your file: {missing}"
bad_levers = [c for c in CONTROLLABLE_LEVERS + SPEND_LEVERS + list(SCENARIO) if c not in NUMERIC_COLUMNS + CATEGORICAL_COLUMNS]
assert not bad_levers, f"Levers must also be listed as feature columns: {bad_levers}"

df = df.dropna(subset=[TARGET_COLUMN])
if TIME_COLUMN and USE_TIME_SPLIT:
    df = df.sort_values(TIME_COLUMN, kind="stable").reset_index(drop=True)   # oldest first
FEATURES = NUMERIC_COLUMNS + CATEGORICAL_COLUMNS
X_all = df[FEATURES].copy()
for c in NUMERIC_COLUMNS:
    X_all[c] = pd.to_numeric(X_all[c], errors="coerce").astype(float)
for c in CATEGORICAL_COLUMNS:
    X_all[c] = X_all[c].astype("string").astype(object)
y_all = pd.to_numeric(df[TARGET_COLUMN], errors="coerce").astype(float)
print(f"Average weekly sales: ${y_all.mean():,.0f}  (middle 80%: ${y_all.quantile(.1):,.0f} – ${y_all.quantile(.9):,.0f})")

## 1. Light EDA: first look at the relationships

In [ ]:
display(pd.DataFrame({"missing_%": X_all.isna().mean() * 100, "unique": X_all.nunique()}).round(1))
corr = X_all[NUMERIC_COLUMNS].corrwith(y_all).sort_values(key=abs, ascending=False)
display(corr.to_frame("correlation_with_sales"))

# Drivers that move together (e.g. staff hours and store size) make individual coefficients
# less certain. Ridge handles this better than plain regression, but it's worth knowing.
pairs = X_all[NUMERIC_COLUMNS].corr().abs().where(lambda m: np.triu(np.ones(m.shape, bool), 1)).stack()
print("Highly related driver pairs (>0.7):", pairs[pairs > 0.7].round(2).to_dict() or "none")

levers_num = [c for c in CONTROLLABLE_LEVERS if c in NUMERIC_COLUMNS][:4]
fig, axes = plt.subplots(1, len(levers_num), figsize=(4.2 * len(levers_num), 3.4))
for ax, col in zip(np.atleast_1d(axes), levers_num):
    ax.scatter(X_all[col], y_all, alpha=0.3, s=10); ax.set_title(f"sales vs {col}")
plt.tight_layout(); plt.show()

## 2. Split and validation plan
For weekly data the honest test is **"train on the past, test on the most recent weeks"**. Cross-validation inside the training period also moves forward in time (`TimeSeriesSplit`). If your rows aren't time-ordered, set `USE_TIME_SPLIT = False`.

In [ ]:
size_mode = DATA_SIZE_MODE if DATA_SIZE_MODE != "auto" else ("small" if len(df) < 1_000 else "large")
if TIME_COLUMN and USE_TIME_SPLIT:
    X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=TEST_SIZE, shuffle=False)
    cv = TimeSeriesSplit(n_splits=5)
    print(f"Time split: train up to {TIME_COLUMN} {df.loc[X_train.index, TIME_COLUMN].max()}, "
          f"test from {df.loc[X_test.index, TIME_COLUMN].min()}")
else:
    X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    cv = (RepeatedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE) if size_mode == "small"
          else KFold(5, shuffle=True, random_state=RANDOM_STATE))
print(f"Data size mode: {size_mode} | train {len(X_train):,} rows, test {len(X_test):,} rows")

## 3. Two leakage-safe pipelines
- **Ridge:** fill gaps → scale numbers → one-hot categories → linear model. Scaling lets us compare drivers fairly.
- **HistGradientBoosting:** fill category gaps → integer codes → trees (no scaling needed).

In [ ]:
ridge_pipe = Pipeline([
    ("prep", ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUMERIC_COLUMNS),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="if_binary"))]),
         CATEGORICAL_COLUMNS),
    ], verbose_feature_names_out=False)),
    ("model", Ridge(alpha=1.0)),
])
hgb_pipe = Pipeline([
    ("prep", ColumnTransformer([
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("codes", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))]),
         CATEGORICAL_COLUMNS),
        ("num", "passthrough", NUMERIC_COLUMNS),
    ], verbose_feature_names_out=False)),
    ("model", HistGradientBoostingRegressor(
        categorical_features=[True] * len(CATEGORICAL_COLUMNS) + [False] * len(NUMERIC_COLUMNS),
        max_leaf_nodes=8, min_samples_leaf=20, learning_rate=0.05, max_iter=300, random_state=RANDOM_STATE)),
])

## 4. Light tuning of Ridge, then compare with the baseline and the boosting cross-check
Ridge's one dial is **alpha**: bigger alpha = coefficients pulled more strongly toward zero (more cautious).
**Simplicity rule:** among alphas within one standard error of the best, take the **largest** (most cautious) one.

In [ ]:
search = GridSearchCV(ridge_pipe, {"model__alpha": [0.1, 0.3, 1, 3, 10]}, cv=cv,
                      scoring="neg_mean_absolute_error", n_jobs=-1).fit(X_train, y_train)
res = pd.DataFrame(search.cv_results_)
best = res.loc[res["rank_test_score"].idxmin()]
se = best["std_test_score"] / np.sqrt(cv.get_n_splits())
chosen_alpha = res.loc[res["mean_test_score"] >= best["mean_test_score"] - se, "param_model__alpha"].astype(float).max()
display(res[["param_model__alpha", "mean_test_score", "std_test_score"]].assign(
    CV_MAE=lambda d: -d["mean_test_score"]).drop(columns="mean_test_score"))
print(f"Best alpha {best['param_model__alpha']} | chosen (most cautious within 1 SE): {chosen_alpha}")
ridge_pipe.set_params(model__alpha=chosen_alpha)

scoring = {"MAE": "neg_mean_absolute_error", "R2": "r2"}
rows = []
for name, m in [("Dummy (average week)", DummyRegressor()), ("Ridge (tuned)", ridge_pipe), ("HistGradientBoosting", hgb_pipe)]:
    r = cross_validate(m, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    rows.append({"model": name, "CV_MAE_$": -r["test_MAE"].mean(), "CV_R2": r["test_R2"].mean()})
comparison = pd.DataFrame(rows)
display(comparison)
gap = comparison.loc[1, "CV_MAE_$"] - comparison.loc[2, "CV_MAE_$"]
print("Ridge is close to boosting, so its simple straight-line story is trustworthy." if gap < 0.05 * comparison.loc[1, "CV_MAE_$"]
      else "Boosting is clearly better, so some effects are curved. Check the shape plots in section 6.")

## 5. Test-set check (most recent weeks)

In [ ]:
ridge_pipe.fit(X_train, y_train); hgb_pipe.fit(X_train, y_train)
for name, m in [("Ridge", ridge_pipe), ("HistGradientBoosting", hgb_pipe)]:
    p = m.predict(X_test)
    print(f"{name:22s} test MAE ${mean_absolute_error(y_test, p):,.0f} | R² {r2_score(y_test, p):.2f}")
print(f"{'Dummy':22s} test MAE ${mean_absolute_error(y_test, np.full(len(y_test), y_train.mean())):,.0f}")

## 6. What drives sales? (the main event)
**Ridge coefficients in plain units.** For each numeric driver we show the effect of **+1 unit** (e.g. +$1 of ad spend, +1 discount point) and of **one typical step** (one standard deviation). For yes/no categories it's the difference versus "no"; for multi-level categories it's versus the average.

In [ ]:
names = ridge_pipe.named_steps["prep"].get_feature_names_out()
coef = pd.Series(ridge_pipe.named_steps["model"].coef_, index=names)
sd = X_train[NUMERIC_COLUMNS].std()
drivers = pd.DataFrame({"effect_per_typical_step_$": coef})
drivers["typical_step"] = [sd.get(n, np.nan) for n in names]
drivers["effect_per_1_unit_$"] = drivers["effect_per_typical_step_$"] / drivers["typical_step"]
drivers["controllable"] = [any(n == lv or n.startswith(lv + "_") for lv in CONTROLLABLE_LEVERS) for n in names]
drivers = drivers.reindex(drivers["effect_per_typical_step_$"].abs().sort_values(ascending=False).index)
display(drivers)

drivers["effect_per_typical_step_$"].iloc[::-1].plot.barh(
    figsize=(7, 5), color=["seagreen" if c else "grey" for c in drivers["controllable"].iloc[::-1]])
plt.title("Effect on weekly sales of one typical step (green = controllable)"); plt.tight_layout(); plt.show()

In [ ]:
# Cross-check with permutation importance on the test weeks (both models). If the top drivers agree,
# you can be more confident in the story.
imp = pd.DataFrame({
    name: permutation_importance(m, X_test, y_test, scoring="neg_mean_absolute_error", n_repeats=15,
                                 random_state=RANDOM_STATE, n_jobs=-1).importances_mean
    for name, m in [("Ridge", ridge_pipe), ("HistGradientBoosting", hgb_pipe)]}, index=FEATURES)
display(imp.sort_values("Ridge", ascending=False).rename(columns=lambda c: f"{c}_MAE_increase_$"))

# Shapes of the controllable levers from the boosting model: flattening curves = diminishing returns
fig, ax = plt.subplots(figsize=(4 * len(levers_num), 3.4))
PartialDependenceDisplay.from_estimator(hgb_pipe, X_train, levers_num, ax=ax, n_cols=len(levers_num))
plt.suptitle("How sales respond to each lever (boosting model)", y=1.05); plt.tight_layout(); plt.show()

## 7. Lever table: what's worth pulling?
For the **spend levers**, return on spend = (extra sales per $1 × gross margin) − $1. Above zero means each extra dollar pays for itself, *on average, at today's spending levels*.

In [ ]:
print("In plain English (Ridge, holding the other drivers fixed):")
for n, row in drivers[drivers["controllable"]].iterrows():
    if n in NUMERIC_COLUMNS:
        line = f"  • +1 {n} → {row['effect_per_1_unit_$']:+,.2f} weekly sales"
        if n in SPEND_LEVERS:
            roi = row["effect_per_1_unit_$"] * GROSS_MARGIN - 1
            line += f"  (return on each extra $1 spent: {roi:+.0%} after margin)"
        print(line)
    else:
        print(f"  • {n} → {row['effect_per_typical_step_$']:+,.0f} weekly sales")

# What-if scenario on the test weeks
X_scn = X_test.copy()
for lever, pct in SCENARIO.items():
    X_scn[lever] = X_scn[lever] * (1 + pct)
base_sales, new_sales = ridge_pipe.predict(X_test).sum(), ridge_pipe.predict(X_scn).sum()
extra_cost = sum((X_scn[l] - X_test[l]).sum() for l in SCENARIO if l in SPEND_LEVERS)
extra_profit = (new_sales - base_sales) * GROSS_MARGIN - extra_cost
print(f"\nWhat-if {SCENARIO} over the test weeks: sales {new_sales - base_sales:+,.0f}, "
      f"extra spend ${extra_cost:,.0f}, net gross profit {extra_profit:+,.0f}")

## 8. What to do next: business recommendations
**Pull harder** on levers with a positive return on spend and a consistent rank in both models (in the demo, digital ads up to the point where the curve flattens, plus promo displays).
**Rethink** levers with a negative return or tiny effect (e.g. local ad spend in the demo): trim and reinvest.
**Price and discounts** trade margin for volume. Run the what-if with your real margin before changing either.

**Prove it:** pick a few matched stores, change one lever for 4–6 weeks, and compare against similar stores that didn't change. Observational data shows what *goes with* higher sales; a test shows what *causes* it.

**Keep it healthy:** refresh quarterly; effects drift as channels saturate and competitors react.

**Small vs. large data**
- *Small (a few hundred store-weeks):* Ridge with a cautious alpha; read effects as ranges, not exact numbers.
- *Large (thousands of store-weeks):* add store and season features, try `log1p` on spend columns to model diminishing returns directly, and let boosting confirm the shapes.

In [ ]:
# Optional: save the driver table for a slide or a BI dashboard
drivers.to_csv("sales_driver_table.csv")
print("Saved sales_driver_table.csv")
top_lever = drivers[drivers["controllable"]].index[0]
print(f"Summary: Ridge test-ready (alpha={chosen_alpha}); strongest controllable lever: {top_lever}.")